### This notebook compute "P2. Rainfall intensity and frequency" indicator for the 27 basins of IKI Project

**Created:** 6/23/2025 by Sophia Bakar (sbakar@rti.org) 

**Project #:** 0219481  

**Last modified:** 12/30/2025 by Sophia

**Status:** complete for baseline and future scenario

**QA Status:** reviewed by  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Peligro\Scripts_Peligro
 
**Objective:** Compute rainfall intensity and frequency indicator 

**Compatibility:** 

**Packages:** numpy, pandas, geopandas, sqlite3, matplotlib, tqdm

**Further documentation:**  
 
**Inputs:**   precipitation from the modeling groups databases

**Outputs:** 
 
**Assumptions:**

**N/A Handling:** Ignores any missing data when calculating percentiles.  
 
**Future work:** 
 
**Notes:** Precipitation is obtained by querying the modeling groups sql databases. For the baseline scenario, we use met_source_id = 2 (source - PISCO) and for the future scenario 8.5 we use met_source_id = 5 (source - CMIP6 85)  

General Methodology:  
1. For each COMID and year, calculates the maximum length of cumulative wet days (CWD) and 95th percentile (95p) of precipitation in mm.  
2. Calculates Quantiles (25th, 50th, and 75th percentiles) for CWD and 95p for each scenario based on longterm data. 
3. Categorizes CWD and 95p for each COMID and scenario into muy alto (very high), alto (high), medio (medium), and bajo (low) based on quantiles.

In [1]:
import numpy as np
import pandas as pd
import sqlite3
import geopandas as gpd
import os
from tqdm import tqdm

In [2]:
# set up user and database path
#user = 'nreynolds'
user = 'sbakar'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"

In [3]:
IndID = 102 #Indicator ID (Peligro = 1 + 0X where X is the Peligro Indicator number) 
scenarios = {
    1: {"met_source_id": 2},  # Baseline
    2: {"met_source_id": 5},  # Future
}

In [4]:
#subbasins_shapefile = fr"C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/GIS_WaterALLOC_General/Peru_AHD_with_districts.shp"
subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index('COMID').to_crs('WGS84')

In [5]:
# threshold for defining the continuous wet days
wet_day_threshold = 0.1 # in mm

# conversion for cm to mm
cm_to_mm = 10.0

In [6]:
# support functions
# get the nmaximum number of continuous wet days in a time series
def compute_cwd(series_mm):
    """Return longest streak of wet days (>0.1 mm)."""
    wet = series_mm > wet_day_threshold
    # Count runs
    max_streak = 0
    current = 0
    for v in wet.values:
        if v:
            current += 1
            if current > max_streak:
                max_streak = current
        else:
            current = 0
    return max_streak

# get total precip in days above the 95th percentile
def compute_R95p(series_mm, p95):
    """Return total precip in days > p95 threshold."""
    return series_mm[series_mm > p95].sum()

# classify the changes between the baseline and comparison periods
def classify_quantile(val, q1, q2, q3):
    """Return qualitative category based on quantiles."""
    if val >= q3:
        return "Muy Alta"
    elif val >= q2:
        return "Alta"
    elif val >= q1:
        return "Media"
    else:
        return "Baja"

In [8]:
records = []   # store all COMID-year results

#this loop processes all groups' databases for both the baseline and future scenarios
for grupo in range(1,13):
    #sqlite_path = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/Grupo_{grupo}/BD/BD_Grupo_{grupo}.sqlite'
    sqlite_path = f"C:/Users/sbakar/OneDrive - Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/Grupo_{grupo}/BD/BD_Grupo_{grupo}.sqlite"
    conn = sqlite3.connect(sqlite_path)

    for ScnID, scn_info in scenarios.items():
      
        query = f"""
        SELECT
            comid,
            avg_precip_cm,
            measured_date
        FROM catchment_met_observations
        WHERE met_source_id = {scn_info['met_source_id']};
        """

        met = pd.read_sql_query(query, conn)

        met['measured_date'] = pd.to_datetime(
            met['measured_date'],
            format='%Y-%m-%d %H:%M:%S %z UTC',
            errors='coerce'
        )
        met = met.dropna(subset=['measured_date'])

        met['precip_mm'] = met['avg_precip_cm'] * cm_to_mm
        met['year'] = met['measured_date'].dt.year

        for (cid, year), group in met.groupby(['comid', 'year']):
            series = group.sort_values('measured_date')['precip_mm']

            records.append({
                'ScnID': ScnID,
                'comid': cid,
                'year': year,
                'CWD': compute_cwd(series),
                'R95p_raw': series
            })

    conn.close()

df = pd.DataFrame(records)

In [9]:
# Build p95 per COMID 
p95_map = {}

for (ScnID, cid), g in df.groupby(['ScnID', 'comid']):
    vals = []
    for s in g['R95p_raw']:
        vals.extend(s)
    vals = np.array(vals)

    p95_map[(ScnID, cid)] = np.nanpercentile(vals, 95) if len(vals) > 0 else np.nan


def calc_R95p(row):
    p95 = p95_map[(row['ScnID'], row['comid'])]
    return row['R95p_raw'][row['R95p_raw'] > p95].sum()


df['R95p'] = df.apply(calc_R95p, axis=1)
df = df.drop(columns=['R95p_raw'])

full_period = df[df['year'].between(1981, 2020)]

longterm_stats = (
    full_period
    .groupby(['ScnID', 'comid'], as_index=False)
    .agg({
        'CWD': 'mean',
        'R95p': 'mean'
    })
    .rename(columns={
        'CWD': 'mean_CWD',
        'R95p': 'mean_R95p'
    })
)

quantiles = {}

for ScnID, g in longterm_stats.groupby('ScnID'):
    quantiles[ScnID] = {
        'CWD': g['mean_CWD'].quantile([0.25, 0.50, 0.75]).values,
        'R95p': g['mean_R95p'].quantile([0.25, 0.50, 0.75]).values
    }

cat_to_val = {"Baja": 1, "Media": 2, "Alta": 3, "Muy Alta": 4}

def apply_classes(row):
    q1, q2, q3 = quantiles[row['ScnID']]['CWD']
    row['CWD_category'] = classify_quantile(row['mean_CWD'], q1, q2, q3)

    q1, q2, q3 = quantiles[row['ScnID']]['R95p']
    row['R95p_category'] = classify_quantile(row['mean_R95p'], q1, q2, q3)

    return row

longterm_stats = longterm_stats.apply(apply_classes, axis=1)

longterm_stats['CWD_value'] = longterm_stats['CWD_category'].map(cat_to_val)
longterm_stats['R95p_value'] = longterm_stats['R95p_category'].map(cat_to_val)



In [10]:
# set up hazard mapping to get final indicator value combining frequency and intensity
hazard_matrix = {
    4: {1: "Medio", 2: "Alto", 3: "Muy alto", 4: "Muy alto"},
    3: {1: "Medio",  2: "Alto", 3: "Alto",     4: "Muy alto"},
    2: {1: "Bajo",  2: "Medio",  3: "Alto",    4: "Alto"},
    1: {1: "Bajo",  2: "Bajo",  3: "Medio",    4: "Medio"},
}

hazard_to_val = {"Bajo": 1,"Medio": 2,"Alto": 3,"Muy alto": 4}

def compute_hazard(freq, intens):
    return hazard_matrix[freq][intens]

In [11]:
longterm_stats['Hazard_category'] = longterm_stats.apply(lambda r: compute_hazard(r['R95p_value'], r['CWD_value']),axis=1)
longterm_stats['Hazard_value'] = longterm_stats['Hazard_category'].map(hazard_to_val)

In [12]:
# Connect to the  SQLite database
#insert the results for both the future and baseline scenarios
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

rows_to_insert = []

for _, row in longterm_stats.iterrows():
    rows_to_insert.append((
        int(row['ScnID']),
        IndID,
        int(row['comid']),
        int(row['Hazard_value'])
    ))

insert_query = """
INSERT OR REPLACE INTO IndValues_Dyn (ScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""

In [13]:
# Check for duplicates in the input dataframe before insert
df_check = pd.DataFrame(rows_to_insert, columns=['ScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['ScnID', 'IndID', 'COMID'])
print("Duplicates in rows_to_insert:", df_check[duplicates])

Duplicates in rows_to_insert: Empty DataFrame
Columns: [ScnID, IndID, COMID, Value]
Index: []


In [14]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()